# TASK 1: CONCEPTUAL ANSWERS

1. What is the difference between "Love" and "love" in NLP?

    *In NLP, 'Love' and 'love' are treated as different tokens due to case sensitivity unless normalized. Lowercasing ensures both are treated the same.*

2. What happens if stopwords are not removed?

    *If stopwords are not removed, models may focus on less meaningful words like 'is', 'the', 'and', which can reduce performance and increase noise.*

3. Provide two real-world scenarios where removing stopwords can be harmful.

    *"Sentiment analysis: 'not good' becomes 'good' if 'not' is removed."*

    *"Question answering: 'What is AI?' loses meaning if 'what' and 'is' are removed."*

4. Explain the difference between stemming and lemmatization.\

    *Stemming cuts words to root form (e.g., 'running' → 'run'), often crudely*
     
    *Lemmatization uses vocabulary and grammar rules (e.g., 'better' → 'good'), producing meaningful base forms*


# NLP PREPROCESSING ENGINE

In [26]:
import re
from collections import Counter
import numpy as np

# Optional: install emoji package if needed
!pip install emoji

import emoji

In [27]:
def preprocess_text(text):
    try:
        if not isinstance(text, str) or text.strip() == "":
            return [], ""

        # Remove URLs
        text = re.sub(r"http\S+|www\S+", "", text)

        # Remove emails
        text = re.sub(r"\S+@\S+", "", text)

        # Remove numbers
        text = re.sub(r"\d+", "", text)

        # Remove emojis
        text = emoji.replace_emoji(text, replace="")

        # Lowercase
        text = text.lower()

        # Handle repeated characters (soooo → soo)
        text = re.sub(r"(.)\1{2,}", r"\1\1", text)

        # Remove special characters
        text = re.sub(r"[^a-z\s]", "", text)

        # Remove extra spaces
        text = re.sub(r"\s+", " ", text).strip()

        tokens = text.split()

        # Remove short tokens (≤2), but keep exceptions
        exceptions = {"no", "not"}
        tokens = [word for word in tokens if len(word) > 2 or word in exceptions]

        cleaned_sentence = " ".join(tokens)

        return tokens, cleaned_sentence

    except Exception as e:
        print(f"Error processing text: {text}")
        return [], ""

# TASK 3: STRESS TESTING

In [28]:
test_sentences = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍😍",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

results = []

print("===== STRESS TEST RESULTS =====\n")

for sentence in test_sentences:
    tokens, cleaned = preprocess_text(sentence)

    print(f"Original: {sentence}")
    print(f"Tokens: {tokens}")
    print(f"Cleaned: {cleaned}")
    print("-" * 50)

    results.append((sentence, tokens, cleaned))


===== STRESS TEST RESULTS =====

Original: Get 100% FREE access now!!!
Tokens: ['get', 'free', 'access', 'now']
Cleaned: get free access now
--------------------------------------------------
Original: I absolutely looooved this product 😍😍
Tokens: ['absolutely', 'looved', 'this', 'product']
Cleaned: absolutely looved this product
--------------------------------------------------
Original: Worst service ever... 0/10
Tokens: ['worst', 'service', 'ever']
Cleaned: worst service ever
--------------------------------------------------
Original: Call me at 9876543210
Tokens: ['call']
Cleaned: call
--------------------------------------------------
Original: This is THE best course!!!
Tokens: ['this', 'the', 'best', 'course']
Cleaned: this the best course
--------------------------------------------------
Original: Visit https://openai.com now!
Tokens: ['visit', 'now']
Cleaned: visit now
--------------------------------------------------
Original: Nooooo this is baaad!!!
Tokens: ['noo', 'this

# TASK 4: TOKEN ANALYTICS

In [29]:
print("\n===== TOKEN ANALYTICS =====\n")

analytics = []

for sentence, tokens, cleaned in results:
    total_tokens = len(tokens)
    unique_tokens = len(set(tokens))
    avg_length = np.mean([len(t) for t in tokens]) if tokens else 0

    analytics.append({
        "sentence": sentence,
        "total_tokens": total_tokens,
        "unique_tokens": unique_tokens,
        "avg_token_length": avg_length
    })

    print(f"Sentence: {sentence}")
    print(f"Total Tokens: {total_tokens}")
    print(f"Unique Tokens: {unique_tokens}")
    print(f"Avg Token Length: {avg_length:.2f}")
    print("-" * 50)


# Identify most noisy sentence (heuristic: biggest reduction)
noise_scores = [(orig, len(orig.split()) - len(tokens)) for orig, tokens, _ in results]
most_noisy = max(noise_scores, key=lambda x: x[1])[0]

# Sentence with most meaningful tokens
most_tokens_sentence = max(analytics, key=lambda x: x["total_tokens"])["sentence"]

print("\nMost noisy sentence:", most_noisy)
print("Most meaningful tokens sentence:", most_tokens_sentence)



===== TOKEN ANALYTICS =====

Sentence: Get 100% FREE access now!!!
Total Tokens: 4
Unique Tokens: 4
Avg Token Length: 4.00
--------------------------------------------------
Sentence: I absolutely looooved this product 😍😍
Total Tokens: 4
Unique Tokens: 4
Avg Token Length: 6.75
--------------------------------------------------
Sentence: Worst service ever... 0/10
Total Tokens: 3
Unique Tokens: 3
Avg Token Length: 5.33
--------------------------------------------------
Sentence: Call me at 9876543210
Total Tokens: 1
Unique Tokens: 1
Avg Token Length: 4.00
--------------------------------------------------
Sentence: This is THE best course!!!
Total Tokens: 4
Unique Tokens: 4
Avg Token Length: 4.25
--------------------------------------------------
Sentence: Visit https://openai.com now!
Total Tokens: 2
Unique Tokens: 2
Avg Token Length: 4.00
--------------------------------------------------
Sentence: Nooooo this is baaad!!!
Total Tokens: 3
Unique Tokens: 3
Avg Token Length: 3.67
------

# TASK 5: FREQUENCY ANALYSIS

In [30]:
from collections import Counter

all_tokens = []
for _, tokens, _ in results:
    all_tokens.extend(tokens)

counter = Counter(all_tokens)

print("\n===== FREQUENCY ANALYSIS =====\n")

print("Top 10 most frequent words:")
print(counter.most_common(10))

print("\nTop 5 least frequent words:")
print(counter.most_common()[:-6:-1])


===== FREQUENCY ANALYSIS =====

Top 10 most frequent words:
[('this', 4), ('now', 3), ('get', 1), ('free', 1), ('access', 1), ('absolutely', 1), ('looved', 1), ('product', 1), ('worst', 1), ('service', 1)]

Top 5 least frequent words:
[('with', 1), ('happy', 1), ('not', 1), ('offer', 1), ('limited', 1)]


# TASK 6: FULL PIPELINE

In [31]:
def full_pipeline(text_list):
    all_tokens = []
    clean_sentences = []

    for text in text_list:
        tokens, cleaned = preprocess_text(text)
        all_tokens.extend(tokens)
        clean_sentences.append(cleaned)

    return {
        "tokens": all_tokens,
        "clean_sentences": clean_sentences
    }

In [32]:
# Example run
pipeline_output = full_pipeline(test_sentences)

print("\n===== FULL PIPELINE OUTPUT =====\n")
print(pipeline_output)


===== FULL PIPELINE OUTPUT =====

{'tokens': ['get', 'free', 'access', 'now', 'absolutely', 'looved', 'this', 'product', 'worst', 'service', 'ever', 'call', 'this', 'the', 'best', 'course', 'visit', 'now', 'noo', 'this', 'baad', 'got', 'win', 'now', 'limited', 'offer', 'not', 'happy', 'with', 'this'], 'clean_sentences': ['get free access now', 'absolutely looved this product', 'worst service ever', 'call', 'this the best course', 'visit now', 'noo this baad', 'got', 'win now limited offer', 'not happy with this']}


# TASK 7: ERROR HANDLING TESTS

In [33]:
edge_cases = ["", "😂😂😂😂", "1234567890"]

print("\n===== EDGE CASE TESTING =====\n")

for case in edge_cases:
    tokens, cleaned = preprocess_text(case)
    print(f"Input: {case}")
    print(f"Tokens: {tokens}")
    print(f"Cleaned: '{cleaned}'")
    print("-" * 50)


===== EDGE CASE TESTING =====

Input: 
Tokens: []
Cleaned: ''
--------------------------------------------------
Input: 😂😂😂😂
Tokens: []
Cleaned: ''
--------------------------------------------------
Input: 1234567890
Tokens: []
Cleaned: ''
--------------------------------------------------
